# didi_workbench_bundle -- notebook runner

**Prefer the terminal route (`scripts/run_stage.sh`) for anything that takes longer than a few minutes.** A browser disconnect kills this notebook's kernel mid-run -- the `pipeline` stage can take well over an hour even at 1% scale (Gate 2's reference 1% run took 1h52m) -- whereas a `nohup` background job started from a Terminal survives a disconnect. Use this notebook for the fast stages (`generate`, `validate`, `schema-check`, `gate` at small scales) or for re-checking results after a terminal-launched `pipeline` run finishes.

Every cell below just calls the same scripts described in `docs/SETUP_GUIDE.md`, via `subprocess` -- there is no logic here that isn't also in `scripts/`. If a cell's behavior looks wrong, the bug is in the referenced script, not here.

## 1. Setup: create the pinned venv and run the estimator's self-test

In [ ]:
import subprocess, sys, os

BUNDLE_ROOT = os.path.dirname(os.getcwd())  # this notebook lives in notebook/, one level below the bundle root

def run(cmd, **kwargs):
    print(">>>", " ".join(cmd))
    result = subprocess.run(cmd, cwd=BUNDLE_ROOT, capture_output=True, text=True, **kwargs)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
    print(f"[exit code {result.returncode}]")
    return result

run(["bash", "scripts/setup_env.sh"])

In [ ]:
run(["bash", "scripts/setup_env.sh", "--self-test"])

## 2. Check the machine

In [ ]:
venv_python = os.path.join(BUNDLE_ROOT, "venv_didi", "bin", "python")
run([venv_python, "scripts/check_machine.py"])

## 3. Small scales via run_stage.sh (fast -- safe to run from a notebook cell)

In [ ]:
SCALE = "0.001"  # change to "0.01", "0.1", "1.0" as you move up the ladder
run(["bash", "scripts/run_stage.sh", "all", SCALE])

## 4. The mandatory Step-3 calibration check

Must pass (or, at very small scales, show only `NOT_PRESENT_AT_THIS_SCALE` rows with the *present* rows passing) before trusting a larger scale's matching load.

In [ ]:
import json
settings = {}
for line in open(os.path.join(BUNDLE_ROOT, "bundle_settings.env")):
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, v = line.split("=", 1)
        settings[k] = v
data_root = os.path.normpath(os.path.join(BUNDLE_ROOT, settings.get("DATA_ROOT", "../didi_data")))
run_dir = os.path.join(data_root, "runs", f"workbench_synthetic_{SCALE}_seed{settings.get('SEED', '42')}")
run([venv_python, "scripts/compare_preflight.py", "--run", run_dir, "--scale", SCALE])

## 5. Build the report for this scale

In [ ]:
run([venv_python, "scripts/make_report.py", "--scale", SCALE])

## 6. For a long `pipeline` stage: launch it from a Terminal instead, then come back here to check status

Open a Terminal (from the Jupyter launcher) and run:
```
bash scripts/run_stage.sh pipeline 0.1
```
That backgrounds the run with `nohup` and survives this notebook's kernel restarting or your browser tab closing. Come back to this cell any time to check on it:

In [ ]:
run(["bash", "scripts/status.sh"])